In [1]:
#euclidean distance between two n-dimensional points

def euclidean_distance(dimension, point1, point2):
    squared_distance = 0
    for i in range(dimension):
        squared_distance += (point1[i] - point2[i]) ** 2
    return squared_distance ** 0.5 



In [2]:
def knn_classifier(k, training_data, test_point):
    distances = []
    for data_point in training_data:
        distance = euclidean_distance(len(test_point), data_point[:-1], test_point)
        distances.append((distance, data_point[-1]))
    distances.sort(key=lambda x: x[0])
    k_nearest_neighbors = distances[:k]
    labels = [neighbor[1] for neighbor in k_nearest_neighbors]
    return max(labels, key=labels.count)

In [3]:
import csv
from pathlib import Path

# Support running from either the notebook folder or the repository root.
candidates = [Path('..'), Path('.'), Path('Homework/Homework1')]
data_folder = next((p for p in candidates if (p / 'Q1_train.csv').is_file()
                    and (p / 'Q1_test.csv').is_file()), None)
if data_folder is None:
    raise FileNotFoundError('Run from Homework1-Solutions, Homework1, or the repository root.')

with open(data_folder / "Q1_train.csv", newline="") as file:
    reader = csv.reader(file)
    next(reader)  # Skip the column names.
    training_data = [[float(value) for value in row] for row in reader]

with open(data_folder / "Q1_test.csv", newline="") as file:
    reader = csv.reader(file)
    next(reader)
    test_data = [[float(value) for value in row] for row in reader]

correct_counts = []
accuracies = []
for k in range(1, 10):  # 1 through 9; the stop value is excluded.
    correct = 0
    for test_row in test_data:
        prediction = knn_classifier(k, training_data, test_row[:-1])
        if prediction == test_row[-1]:
            correct += 1
    correct_counts.append(correct)
    accuracies.append(correct / len(test_data))

print(f"Training rows: {len(training_data)}, test rows: {len(test_data)}")
print(f"{'K':<22}" + "".join(f"{k:>8}" for k in range(1, 10)))
print(f"{'Correct predictions':<22}" + "".join(f"{count:>8}" for count in correct_counts))
print(f"{'Accuracy (proportion)':<22}" + "".join(f"{accuracy:>8.3f}" for accuracy in accuracies))


Training rows: 910, test rows: 462
K                            1       2       3       4       5       6       7       8       9
Correct predictions        462     462     462     462     462     462     462     462     462
Accuracy (proportion)    1.000   1.000   1.000   1.000   1.000   1.000   1.000   1.000   1.000


In [4]:
def min_max_normalize(data, feature_ranges=None):
    # When no ranges are supplied, calculate them from the training data.
    if feature_ranges is None:
        feature_ranges = []
        for i in range(len(data[0]) - 1):  # Exclude the final label column.
            column = [row[i] for row in data]
            feature_ranges.append((min(column), max(column)))

    normalized_data = []
    for row in data:
        normalized_row = []
        for i in range(len(row) - 1):
            min_value, max_value = feature_ranges[i]
            if max_value == min_value:
                normalized_value = 0.0  # Ignore a constant training feature.
            else:
                normalized_value = (row[i] - min_value) / (max_value - min_value)
            normalized_row.append(normalized_value)
        normalized_row.append(row[-1])  # Keep the original class label.
        normalized_data.append(normalized_row)

    return normalized_data, feature_ranges


In [5]:
def standardize(data, feature_stats=None):
    # When no stats are supplied, calculate them from the training data.
    if feature_stats is None:
        feature_stats = []
        for i in range(len(data[0]) - 1):  # Exclude the final label column.
            column = [row[i] for row in data]
            mean = sum(column) / len(column)
            variance = sum((x - mean) ** 2 for x in column) / len(column)
            std_dev = variance ** 0.5
            feature_stats.append((mean, std_dev))

    standardized_data = []
    for row in data:
        standardized_row = []
        for i in range(len(row) - 1):
            mean, std_dev = feature_stats[i]
            if std_dev == 0:
                standardized_value = 0.0  # Ignore a constant training feature.
            else:
                standardized_value = (row[i] - mean) / std_dev
            standardized_row.append(standardized_value)
        standardized_row.append(row[-1])  # Keep the original class label.
        standardized_data.append(standardized_row)

    return standardized_data, feature_stats

In [6]:
# Calculate scaling parameters from the original training data only.
normalized_train, feature_ranges = min_max_normalize(training_data)
normalized_test, _ = min_max_normalize(test_data, feature_ranges)

normalized_correct_counts = []
normalized_accuracies = []
for k in range(1, 10):
    correct = 0
    for test_row in normalized_test:
        prediction = knn_classifier(k, normalized_train, test_row[:-1])
        if prediction == test_row[-1]:
            correct += 1
    normalized_correct_counts.append(correct)
    normalized_accuracies.append(correct / len(normalized_test))

print("Min-max normalized data")
print(f"{'K':<22}" + "".join(f"{k:>8}" for k in range(1, 10)))
print(f"{'Correct predictions':<22}" + "".join(f"{count:>8}" for count in normalized_correct_counts))
print(f"{'Accuracy (proportion)':<22}" + "".join(f"{accuracy:>8.3f}" for accuracy in normalized_accuracies))


Min-max normalized data
K                            1       2       3       4       5       6       7       8       9
Correct predictions        461     461     461     461     461     461     461     461     461
Accuracy (proportion)    0.998   0.998   0.998   0.998   0.998   0.998   0.998   0.998   0.998


In [7]:
# Calculate scaling parameters from the original training data only.
standardized_train, feature_stats = standardize(training_data)
standardized_test, _ = standardize(test_data, feature_stats)

standardized_correct_counts = []
standardized_accuracies = []
for k in range(1, 10):
    correct = 0
    for test_row in standardized_test:
        prediction = knn_classifier(k, standardized_train, test_row[:-1])
        if prediction == test_row[-1]:
            correct += 1
    standardized_correct_counts.append(correct)
    standardized_accuracies.append(correct / len(standardized_test))

print("Standardized data")
print(f"{'K':<22}" + "".join(f"{k:>8}" for k in range(1, 10)))
print(f"{'Correct predictions':<22}" + "".join(f"{count:>8}" for count in standardized_correct_counts))
print(f"{'Accuracy (proportion)':<22}" + "".join(f"{accuracy:>8.3f}" for accuracy in standardized_accuracies))


Standardized data
K                            1       2       3       4       5       6       7       8       9
Correct predictions        461     461     461     461     461     461     461     461     461
Accuracy (proportion)    0.998   0.998   0.998   0.998   0.998   0.998   0.998   0.998   0.998


In [8]:
# Keep a record whenever either scaled version misclassifies a test point.
misclassified_points = []
for k in range(1, 10):
    for i, test_row in enumerate(test_data):
        true_label = test_row[-1]
        minmax_prediction = knn_classifier(k, normalized_train, normalized_test[i][:-1])
        standardized_prediction = knn_classifier(k, standardized_train, standardized_test[i][:-1])

        if minmax_prediction != true_label or standardized_prediction != true_label:
            original_prediction = knn_classifier(k, training_data, test_row[:-1])
            misclassified_points.append({
                "k": k,
                "test_index": i,  # Python indices start at 0.
                "csv_line": i + 2,  # Account for the header and 1-based line numbers.
                "original_features": test_row[:-1],
                "normalized_features": normalized_test[i][:-1],
                "standardized_features": standardized_test[i][:-1],
                "true_label": true_label,
                "original_prediction": original_prediction,
                "minmax_prediction": minmax_prediction,
                "standardized_prediction": standardized_prediction,
            })

for point in misclassified_points:
    print(f"K={point['k']}, test index={point['test_index']}, CSV line={point['csv_line']}")
    print("  Original features:", point["original_features"])
    print(f"  True label: {point['true_label']:g}; original prediction: {point['original_prediction']:g}; "
          f"min-max prediction: {point['minmax_prediction']:g}; standardized prediction: {point['standardized_prediction']:g}")

if not misclassified_points:
    print("Neither scaled version misclassified any test points.")


K=1, test index=194, CSV line=196
  Original features: [0.74054, 0.36625, 2.1992, 0.48403]
  True label: 0; original prediction: 0; min-max prediction: 1; standardized prediction: 1
K=2, test index=194, CSV line=196
  Original features: [0.74054, 0.36625, 2.1992, 0.48403]
  True label: 0; original prediction: 0; min-max prediction: 1; standardized prediction: 1
K=3, test index=194, CSV line=196
  Original features: [0.74054, 0.36625, 2.1992, 0.48403]
  True label: 0; original prediction: 0; min-max prediction: 1; standardized prediction: 1
K=4, test index=194, CSV line=196
  Original features: [0.74054, 0.36625, 2.1992, 0.48403]
  True label: 0; original prediction: 0; min-max prediction: 1; standardized prediction: 1
K=5, test index=194, CSV line=196
  Original features: [0.74054, 0.36625, 2.1992, 0.48403]
  True label: 0; original prediction: 0; min-max prediction: 1; standardized prediction: 1
K=6, test index=194, CSV line=196
  Original features: [0.74054, 0.36625, 2.1992, 0.48403]

In [9]:
# Inspect each distinct misclassified point only once.
test_indices = sorted(set(point["test_index"] for point in misclassified_points))
configurations = [
    ("Original", training_data, test_data),
    ("Min-max normalized", normalized_train, normalized_test),
    ("Standardized", standardized_train, standardized_test),
]

for test_index in test_indices:
    print(f"\nTest index {test_index}, test CSV line {test_index + 2}")
    print("Original features:", test_data[test_index][:-1])
    print("True label:", test_data[test_index][-1])

    for name, train_rows, test_rows in configurations:
        neighbors = []
        test_features = test_rows[test_index][:-1]
        for train_index, train_row in enumerate(train_rows):
            distance = euclidean_distance(len(test_features), train_row[:-1], test_features)
            neighbors.append((distance, train_row[-1], train_index))
        neighbors.sort(key=lambda neighbor: neighbor[0])

        print(f"\n{name}")
        print("Test features in this coordinate system:", test_features)
        for k in range(1, 10):
            nearest = neighbors[:k]
            labels = [neighbor[1] for neighbor in nearest]
            prediction = max(labels, key=labels.count)
            print(f"\nK={k}: class 0 votes={labels.count(0)}, class 1 votes={labels.count(1)}, prediction={prediction:g}")
            print(f"{'Rank':>4} {'Train index':>11} {'CSV line':>8} {'Distance':>10} {'Label':>6}  Training features in this coordinate system")
            for rank, (distance, label, train_index) in enumerate(nearest, start=1):
                print(f"{rank:>4} {train_index:>11} {train_index + 2:>8} {distance:>10.6f} {label:>6g}  {train_rows[train_index][:-1]}")
            if labels.count(0) == labels.count(1):
                print("Tie: the nearest neighbor's label wins.")



Test index 194, test CSV line 196
Original features: [0.74054, 0.36625, 2.1992, 0.48403]
True label: 0.0

Original
Test features in this coordinate system: [0.74054, 0.36625, 2.1992, 0.48403]

K=1: class 0 votes=1, class 1 votes=0, prediction=0
Rank Train index CSV line   Distance  Label  Training features in this coordinate system
   1         170      172   1.480617      0  [0.5706, -0.0248, 1.2421, -0.5621]

K=2: class 0 votes=2, class 1 votes=0, prediction=0
Rank Train index CSV line   Distance  Label  Training features in this coordinate system
   1         170      172   1.480617      0  [0.5706, -0.0248, 1.2421, -0.5621]
   2         226      228   1.480617      0  [0.5706, -0.0248, 1.2421, -0.5621]

K=3: class 0 votes=3, class 1 votes=0, prediction=0
Rank Train index CSV line   Distance  Label  Training features in this coordinate system
   1         170      172   1.480617      0  [0.5706, -0.0248, 1.2421, -0.5621]
   2         226      228   1.480617      0  [0.5706, -0.0248

In [10]:
# Exploratory comparison: keep normalized F1-F3 and the class label.
# row[:3] selects F1-F3; [row[-1]] keeps the label as a one-item list.
normalized_train_no_f4 = [row[:3] + [row[-1]] for row in normalized_train]
normalized_test_no_f4 = [row[:3] + [row[-1]] for row in normalized_test]

no_f4_correct_counts = []
no_f4_accuracies = []
no_f4_error_indices = {}
for k in range(1, 10):
    correct = 0
    error_indices = []
    for i, test_row in enumerate(normalized_test_no_f4):
        prediction = knn_classifier(k, normalized_train_no_f4, test_row[:-1])
        if prediction == test_row[-1]:
            correct += 1
        else:
            error_indices.append(i)
    no_f4_correct_counts.append(correct)
    no_f4_accuracies.append(correct / len(normalized_test_no_f4))
    no_f4_error_indices[k] = error_indices

print("Min-max normalized data, feature 4 dropped (exploratory)")
print(f"{'K':<22}" + "".join(f"{k:>8}" for k in range(1, 10)))
print(f"{'Correct predictions':<22}" + "".join(f"{count:>8}" for count in no_f4_correct_counts))
print(f"{'Accuracy (proportion)':<22}" + "".join(f"{accuracy:>8.3f}" for accuracy in no_f4_accuracies))
print("Misclassified test indices (0-based):", no_f4_error_indices)


Min-max normalized data, feature 4 dropped (exploratory)
K                            1       2       3       4       5       6       7       8       9
Correct predictions        462     462     462     462     462     462     462     462     461
Accuracy (proportion)    1.000   1.000   1.000   1.000   1.000   1.000   1.000   1.000   0.998
Misclassified test indices (0-based): {1: [], 2: [], 3: [], 4: [], 5: [], 6: [], 7: [], 8: [], 9: [402]}


In [11]:
from itertools import combinations

# Exploratory: these test-set comparisons are not independent validation.
# Enumerate features we KEEP: 4 + 6 + 4 + 1 = 15 nonempty subsets.
subset_results = []
scaling_versions = [
    ("Original", training_data, test_data),
    ("Min-max", normalized_train, normalized_test),
    ("Standardized", standardized_train, standardized_test),
]
for size in range(1, 5):
    for features in combinations(range(4), size):
        for strategy, train_rows, test_rows in scaling_versions:
            selected_train = [[row[i] for i in features] + [row[-1]] for row in train_rows]
            selected_test = [[row[i] for i in features] + [row[-1]] for row in test_rows]
            correct_counts_subset = [0] * 9

            for test_row in selected_test:
                distances = []
                for train_row in selected_train:
                    distance = euclidean_distance(size, train_row[:-1], test_row[:-1])
                    distances.append((distance, train_row[-1]))
                distances.sort(key=lambda neighbor: neighbor[0])
                # The same sorted neighbors serve all nine K values.
                for k in range(1, 10):
                    labels = [neighbor[1] for neighbor in distances[:k]]
                    prediction = max(labels, key=labels.count)
                    if prediction == test_row[-1]:
                        correct_counts_subset[k - 1] += 1

            for k, correct in enumerate(correct_counts_subset, start=1):
                subset_results.append({
                    "features": tuple(i + 1 for i in features),
                    "strategy": strategy,
                    "k": k,
                    "correct": correct,
                    "accuracy": correct / len(selected_test),
                })

# Match the original all-feature result: every test point classified correctly.
perfect_results = [result for result in subset_results if result["correct"] == len(test_data)]
print(f"Explored {len(subset_results)} configurations. Showing only those matching 100% test accuracy.")
print("Features listed are KEPT; all four features are included as a reference.")
print(f"{'Features':<16} {'Scaling':<14} K values with 100% accuracy")
for size in range(1, 5):
    for features in combinations(range(1, 5), size):
        for strategy, _, _ in scaling_versions:
            matching_k = [result["k"] for result in perfect_results
                          if result["features"] == features and result["strategy"] == strategy]
            if matching_k:
                feature_names = ", ".join(f"F{i}" for i in features)
                print(f"{feature_names:<16} {strategy:<14} {', '.join(str(k) for k in matching_k)}")

if perfect_results:
    print("Smallest perfect subset size:", min(len(result["features"]) for result in perfect_results))


Explored 405 configurations. Showing only those matching 100% test accuracy.
Features listed are KEPT; all four features are included as a reference.
Features         Scaling        K values with 100% accuracy
F1, F2, F3       Original       1, 2, 3, 4, 5, 6
F1, F2, F3       Min-max        1, 2, 3, 4, 5, 6, 7, 8
F1, F2, F3       Standardized   1, 2, 3, 4, 5, 6, 7, 8, 9
F1, F2, F3, F4   Original       1, 2, 3, 4, 5, 6, 7, 8, 9
Smallest perfect subset size: 3
